In [ ]:
"""Dummy Amazon Connect Contact Lens data generator (Databricks notebook).

Generates synthetic "analysis_redacted" JSON records matching the schema
bronze_notebook.py expects, and writes them directly to the S3 raw-landing
path (or the equivalent Unity Catalog volume path, if configured), laid out
per region as:

    <raw_landing_root>/<REGION>/<yyyy>/<mm>/<dd>/analysis_redacted-<contact_id>.json

Runs entirely inside Databricks - no local AWS CLI or credentials needed,
since it uses the same storage credential already wired up for the pipeline.

Parameters (widgets):
    num_records : total number of dummy calls to generate, split evenly
                   across the given regions (for today's date only)
    regions     : comma-separated region codes, e.g. "EU,NAM,APAC"
                  (must match entries in config/pipeline_config.yaml -> regions)
    env         : "dev" | "stg" | "prod" (defaults to "dev")
"""

In [ ]:
%run ../config/config_loader

In [ ]:
# Imports
import json
import random
import uuid
from datetime import datetime

In [ ]:
# Runtime parameters
dbutils.widgets.text("env", "dev", "Environment (dev/stg/prod)")
dbutils.widgets.text("num_records", "20", "Number of dummy records to create")
dbutils.widgets.text("regions", "EU", "Comma-separated regions (e.g. EU,NAM,APAC)")

env = dbutils.widgets.get("env") or "dev"
num_records = int(dbutils.widgets.get("num_records") or 20)
regions = [r.strip().upper() for r in dbutils.widgets.get("regions").split(",") if r.strip()]

if not regions:
    dbutils.notebook.exit("[ERROR] No regions provided")

print("DUMMY DATA GENERATION STARTED")
print(f"Environment    : {env}")
print(f"Total records  : {num_records}")
print(f"Regions        : {regions}")

In [ ]:
# Resolve the raw-landing root from config (works whether it's an s3:// URI
# or a /Volumes/... path - dbutils.fs handles both transparently)
cfg = load_pipeline_config(env=env, config_path="../config/pipeline_config.yaml")

valid_regions = set(cfg["regions"])
unknown = [r for r in regions if r not in valid_regions]
if unknown:
    print(f"[WARN] Unknown region(s) not in pipeline_config.yaml: {unknown} (proceeding anyway)")

RAW_LANDING_ROOT = resolve_raw_landing_root(cfg)
print(f"Raw landing root: {RAW_LANDING_ROOT}")

In [ ]:
# Sample transcript content used to build fake conversations
SAMPLE_LINES = [
    "Hello, thank you for calling support, how can I help you today?",
    "Hi, I'm having trouble with my recent order, it hasn't arrived yet.",
    "I'm sorry to hear that. Can you provide your order number?",
    "Sure, it's ORD-{n}.",
    "Let me check that for you, one moment please.",
    "I can see the order is currently in transit and should arrive within two days.",
    "Okay, thank you for checking.",
    "Is there anything else I can help you with today?",
    "No, that's all. Thanks for your help!",
    "You're welcome, have a great day!",
]

PARTICIPANT_ROLES = ["AGENT", "CUSTOMER"]


def _build_transcript(contact_id: str, num_lines: int) -> list:
    """Build a fake alternating agent/customer transcript with offsets."""
    transcript = []
    offset = 0
    for i in range(num_lines):
        role = PARTICIPANT_ROLES[i % 2]
        duration = random.randint(2000, 6000)
        line = SAMPLE_LINES[i % len(SAMPLE_LINES)].format(n=random.randint(1000, 9999))
        transcript.append(
            {
                "ParticipantId": f"{role}_1",
                "Id": f"{contact_id}-line-{i+1}",
                "BeginOffsetMillis": offset,
                "EndOffsetMillis": offset + duration,
                "Content": line,
                "Sentiment": random.choice(["POSITIVE", "NEUTRAL", "NEGATIVE"]),
                "LoudnessScore": [round(random.uniform(30, 80), 2) for _ in range(2)],
                "ActionItemsDetected": None,
                "IssuesDetected": None,
                "OutcomesDetected": None,
                "Redaction": {"RedactedTimestamps": []},
            }
        )
        offset += duration
    return transcript


def _build_conversation_characteristics(total_duration_millis: int) -> dict:
    """Build fake sentiment/talk-time/talk-speed metrics for one call."""
    return {
        "TotalConversationDurationMillis": total_duration_millis,
        "TalkSpeed": {
            "DetailsByParticipant": {
                "AGENT": {"AverageWordsPerMinute": round(random.uniform(120, 160), 1)},
                "CUSTOMER": {"AverageWordsPerMinute": round(random.uniform(100, 150), 1)},
            }
        },
        "TalkTime": {
            "TotalTimeMillis": int(total_duration_millis * 0.8),
            "DetailsByParticipant": {
                "AGENT": {"TotalTimeMillis": int(total_duration_millis * 0.45)},
                "CUSTOMER": {"TotalTimeMillis": int(total_duration_millis * 0.35)},
            },
        },
        "NonTalkTime": {"TotalTimeMillis": int(total_duration_millis * 0.2)},
        "Sentiment": {
            "OverallSentiment": {
                "AGENT": round(random.uniform(1, 5), 2),
                "CUSTOMER": round(random.uniform(1, 5), 2),
            },
            "SentimentByPeriod": {
                "QUARTER": {
                    "AGENT": [
                        {"EndOffsetMillis": int(total_duration_millis * q / 4), "Score": round(random.uniform(1, 5), 2)}
                        for q in range(1, 5)
                    ],
                    "CUSTOMER": [
                        {"EndOffsetMillis": int(total_duration_millis * q / 4), "Score": round(random.uniform(1, 5), 2)}
                        for q in range(1, 5)
                    ],
                }
            },
        },
    }


def build_dummy_record(region: str) -> dict:
    """Build one fake Contact Lens 'analysis_redacted' record for the given region."""
    contact_id = str(uuid.uuid4())
    num_lines = random.randint(6, 10)
    transcript = _build_transcript(contact_id, num_lines)
    total_duration = transcript[-1]["EndOffsetMillis"] if transcript else 0

    return {
        "AccountId": "111122223333",
        "CustomerMetadata": {
            "InputS3Uri": f"s3://dummy-connect-recordings/{region}/{contact_id}.wav",
            "InstanceId": f"dummy-instance-{region.lower()}",
            "ContactId": contact_id,
        },
        "JobStatus": "COMPLETED",
        "LanguageCode": "en-US",
        "Transcript": transcript,
        "Participants": [
            {"ParticipantId": "AGENT_1", "ParticipantRole": "AGENT"},
            {"ParticipantId": "CUSTOMER_1", "ParticipantRole": "CUSTOMER"},
        ],
        "Categories": {
            "MatchedCategories": random.sample(
                ["Billing", "TechnicalIssue", "Complaint", "Compliment", "Escalation"], k=random.randint(0, 2)
            )
        },
        "ConversationCharacteristics": _build_conversation_characteristics(total_duration),
    }

In [ ]:
# Split num_records evenly across the given regions (remainder goes to the
# first regions in the list)
base_count, remainder = divmod(num_records, len(regions))
records_per_region = {
    region: base_count + (1 if i < remainder else 0) for i, region in enumerate(regions)
}
print(f"Records per region: {records_per_region}")

In [ ]:
# Generate and write files directly to the raw-landing path via dbutils.fs
today = datetime.today().date()
date_subpath = f"{today.year}/{today.month:02d}/{today.day:02d}"

total_written = 0
for region, count in records_per_region.items():
    region_path = f"{RAW_LANDING_ROOT}/{region}/{date_subpath}/"
    print(f"Writing {count} dummy records to {region_path}")

    for _ in range(count):
        record = build_dummy_record(region)
        contact_id = record["CustomerMetadata"]["ContactId"]
        file_path = f"{region_path}analysis_redacted-{contact_id}.json"

        payload = json.dumps(record, indent=2)
        dbutils.fs.put(file_path, payload, overwrite=True)
        total_written += 1

print(f"DUMMY DATA GENERATION COMPLETE - {total_written} files written across {len(regions)} region(s)")

In [ ]:
# Quick sanity check: list what was just written per region
for region in records_per_region:
    listing_path = f"{RAW_LANDING_ROOT}/{region}/{date_subpath}/"
    try:
        files = dbutils.fs.ls(listing_path)
        print(f"{region}: {len(files)} file(s) at {listing_path}")
    except Exception as exc:
        print(f"[WARN] Could not list {listing_path}: {exc}")